<a href="https://colab.research.google.com/github/Gauravd1710/Deep_Learning_1/blob/main/01_data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/LegalDocAnalyzer'

Mounted at /content/drive


In [2]:
import os

folders = [
    f'{BASE}/data/raw',
    f'{BASE}/data/annotated',
    f'{BASE}/data/processed',
    f'{BASE}/models/ner/pytorch',
    f'{BASE}/models/segmenter',
    f'{BASE}/notebooks',
    f'{BASE}/src',
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f'✓ {folder}')

✓ /content/drive/MyDrive/LegalDocAnalyzer/data/raw
✓ /content/drive/MyDrive/LegalDocAnalyzer/data/annotated
✓ /content/drive/MyDrive/LegalDocAnalyzer/data/processed
✓ /content/drive/MyDrive/LegalDocAnalyzer/models/ner/pytorch
✓ /content/drive/MyDrive/LegalDocAnalyzer/models/segmenter
✓ /content/drive/MyDrive/LegalDocAnalyzer/notebooks
✓ /content/drive/MyDrive/LegalDocAnalyzer/src


In [3]:
!pip install transformers datasets seqeval wandb pymupdf onnx onnxruntime -q
!pip install -U spacy -q
!python -m spacy download en_core_web_trf -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 82.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 47.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [4]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

GPU available: True
GPU name: Tesla T4


In [5]:
import transformers
import datasets
import seqeval
import spacy
import fitz        # PyMuPDF
import onnx
import onnxruntime
import wandb

print("transformers :", transformers.__version__)
print("datasets     :", datasets.__version__)
print("spacy        :", spacy.__version__)
print("torch        :", torch.__version__)
print("onnxruntime  :", onnxruntime.__version__)
print("\n✅ All libraries loaded successfully")

transformers : 5.0.0
datasets     : 4.0.0
spacy        : 3.8.11
torch        : 2.10.0+cu128
onnxruntime  : 1.24.3

✅ All libraries loaded successfully


In [6]:
schemas_code = '''
from dataclasses import dataclass, asdict, field
from typing import List

@dataclass
class Entity:
    text: str
    label: str
    start: int
    end: int
    confidence: float

@dataclass
class Clause:
    clause_id: int
    text: str
    page: int
    word_count: int
    entities: List[Entity] = field(default_factory=list)

    def to_dict(self):
        return asdict(self)

@dataclass
class ContractAnalysis:
    status: str
    total_clauses: int
    clauses: List[Clause]

    def to_dict(self):
        return asdict(self)
'''

with open(f'{BASE}/src/schemas.py', 'w') as f:
    f.write(schemas_code)

print("✅ schemas.py saved to Drive")

✅ schemas.py saved to Drive


In [9]:
import wandb
wandb.login('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')  # paste your API key from wandb.ai/authorize

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [10]:
wandb.init(project='legal-ner', name='setup-test', config={
    'model': 'legal-bert-base-uncased',
    'task': 'NER',
    'team_member': 'Gaurav'
})
wandb.finish()
print("✅ W&B project created")

✅ W&B project created


In [12]:
reqs = """torch
transformers
datasets
seqeval
wandb
pymupdf
onnx
onnxruntime
spacy
numpy
pandas
scikit-learn
matplotlib
tqdm
python-dotenv
"""

with open(f'{BASE}/requirements.txt', 'w') as f:
    f.write(reqs)

print("✅ requirements.txt saved")

✅ requirements.txt saved
